### 1. Calculo de metricas
Para este notebook se requiere utilizar hasta Python 3.10, para que la libreria AligScore se deben contar con versiones especificas que pueden hacer funcionar mal los procesos de fine tuning por lo que se dejan por separado, la idea es tomar todos los archivos csv que cuentan con el texto cientifico, texto resumen original y el resumen generado por medio del LLM en este notebook y realizar el calculo de las metricas: Legibilidad, Relevancia y Factualidad.

Instalacion libreria AlignScore:
https://github.com/yuh-zha/AlignScore

In [1]:
!pip install --quiet  -r req-fine-models-metrics.txt

In [1]:
import pandas as pd
import numpy as np
import textstat
from typing import List, Dict, Any, Optional, Tuple
from bert_score import score as bert_score
import torch
from pathlib import Path
DATA_ALIGN = Path("./models/alignscore")
DATA_ALIGN.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
### Modelo requerido base, puede utilizarse large tambien, podria descargarse de HuggingFace, en una proxima revision lo ajusto.

#!(cd models/alignscore; curl -O -OL https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt)
!cd models/alignscore
!curl -O -OL https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-large.ckpt


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  1321  100  1321    0     0   5189      0 --:--:-- --:--:-- --:--:--  5262

  0 4668M    0 9425k    0     0  10.0M      0  0:07:42 --:--:--  0:07:42 10.0M
  1 4668M    1 54.1M    0     0  28.3M      0  0:02:44  0:00:01  0:02:43 45.0M
  1 4668M    1 85.6M    0     0  29.4M      0  0:02:38  0:00:02  0:02:36 38.2M
  2 4668M    2  114M    0     0  29.1M      0  0:02:40  0:00:03  0:02:37 34.9M
  3 4668M    3  143M    0     0  29.1M      0  0:02:40  0:00:04  0:02:36 33.5M
  3 4668M    3  164M    0     0  27.7M      0  0:02:48  0:00:05  0:02:43 31.0M
  4 4668M    4  191M    0     0  27.6M      0  0:02:48  0:00:06  0:02:42 27.4M
  4 4668M    4  216M    0     0  27.3M      0  0:02:50  0:00:07  0:02:43 26.1M
  5 4668M    5  234M    0     0  26.3M      0  0:0

In [3]:
def calcular_factualidad_alignscore(preds, refs,evaluation_mode, batch_size, device,flag_threshold: float = 0.5):
#evaluation_mode,    # 'nli_sp' (por defecto AlignScore), 'nli', 'bin_sp', 'bin'
    assert len(preds) == len(refs), "preds y refs deben tener la misma longitud"

    # Import tardío para que esta función siga importando aunque no esté instalada la lib.
    from alignscore import AlignScore  
    # Inicializar scorer
    backbone = 'roberta-base'
    scorer = AlignScore(
        model="roberta-base",
        batch_size=batch_size,
        device=device,
        ckpt_path='models/alignscore/AlignScore-large.ckpt',
        evaluation_mode=evaluation_mode
    )

    scores = scorer.score(contexts=refs, claims=preds) 
    scores = [float(s) for s in scores]

    flags_low = [bool(s < flag_threshold) for s in scores]
    per_example = pd.DataFrame({"alignscore": scores,"flag_low": flags_low}) 

    summary = {
        "mean_alignscore": float(np.mean(scores)) if scores else float("nan"),
        "std_alignscore":  float(np.std(scores)) if scores else float("nan"),
        "min_alignscore":  float(np.min(scores)) if scores else float("nan"),
        "max_alignscore":  float(np.max(scores)) if scores else float("nan"),
        "n_examples":      int(len(scores)),
        "backbone":        backbone,
        "evaluation_mode": evaluation_mode,
        "batch_size":      int(batch_size),
        "device":          device,
        "ckpt_path":       'models/alignscore/AlignScore-base.ckpt',
        "flag_threshold":  float(flag_threshold)
    }

    return summary, per_example



In [4]:
def calcular_bertscore_relevancia(preds,refs,idf,rescale_with_baseline,batch_size,device):

    modelo = "roberta-base"
    assert len(preds) == len(refs), "preds y refs deben tener la misma longitud"


    P, R, F1 = bert_score(
        cands=preds.tolist(),
        refs=refs.tolist(),
        lang='en',
        model_type=modelo,
        idf=idf,
        rescale_with_baseline=rescale_with_baseline,
        batch_size=batch_size,
        device=device
    )

    p_list = [float(p) for p in P]
    r_list = [float(r) for r in R]
    f1_list = [float(f) for f in F1]

    summary = {
        "mean_precision": float(np.mean(p_list)) if p_list else float("nan"),
        "mean_recall":    float(np.mean(r_list)) if r_list else float("nan"),
        "mean_f1":        float(np.mean(f1_list)) if f1_list else float("nan"),
        "backbone_for_bertscore": modelo,
        "idf": bool(idf),
        "rescale_with_baseline": bool(rescale_with_baseline),
        "batch_size": int(batch_size),
        "device": device if device is not None else "auto"
    }

    per_example = {
        "bertscore_precision": p_list,
        "bertscore_recall": r_list,
        "bertscore_f1": f1_list
    }

    per_example = pd.DataFrame(per_example)

    return summary, per_example


In [5]:
def calcular_legibilidad_textstat(preds):
    lang = 'en'
    textstat.set_lang(lang)
    rows = []
    for t in preds:
        t = t or ""

        row = {
            "flesch_reading_ease":  float(textstat.flesch_reading_ease(t)),
            "flesch_kincaid_grade": float(textstat.flesch_kincaid_grade(t)),
        }
        row.update({
            "gunning_fog":              float(textstat.gunning_fog(t)),
            "smog_index":               float(textstat.smog_index(t)) if textstat.sentence_count(t) >= 3 else float("nan"),
            "dale_chall":               float(textstat.dale_chall_readability_score(t)),
            "automated_readability":    float(textstat.automated_readability_index(t)),
            "coleman_liau":             float(textstat.coleman_liau_index(t)),
            "text_standard":            textstat.text_standard(t, float_output=True),
            "num_sentences":            int(textstat.sentence_count(t)),
            "num_words":                int(textstat.lexicon_count(t, removepunct=True)),
            "syllables":                int(textstat.syllable_count(t)),
            "reading_time_sec":         float(textstat.reading_time(t)),
        })

        rows.append(row)

    def _try_mean(key: str):
        vals = [r[key] for r in rows if key in r and isinstance(r[key], (int, float)) and not np.isnan(r[key])]
        return float(np.mean(vals)) if vals else float("nan")

    keys = sorted({k for r in rows for k in r.keys()})
    summary = {"n_examples": len(preds), "lang": lang}
    for k in keys:
        summary[f"mean_{k}"] = _try_mean(k)

    per_example = pd.DataFrame(rows) 
    return summary, per_example


In [6]:
def calcular_metricas(df):
    print('**** Inicio relevancia')
    summary_relevancia, per_example_relevancia = calcular_bertscore_relevancia(
        df['gen_summary'],df['article'],
        idf=True,#Set pequeno a false, sino dejar en TRUE
        rescale_with_baseline=False,
        batch_size=1,
        device=device
    )
    print('**** Fin relevancia')
    print('**** Inicio legibilidad')
    summary_legibilidad, per_example_legibilidad = calcular_legibilidad_textstat(df['gen_summary'])
    print('**** Fin legibilidad')
    print('**** Inicio factualidad')
    summary_factualidad, per_example_factualidad = calcular_factualidad_alignscore(df['gen_summary'], df['article'],'nli_sp', 16, device)
    print('**** Fin factualidad')
    return summary_relevancia, summary_legibilidad,summary_factualidad


### Calculo de ejemplo de las 3 metricas requeridas para llama3

In [38]:
data = pd.read_csv('models/results/summaries_llama3.2-1b.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)

**** Inicio relevancia


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, 

**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file models/alignscore/AlignScore-base.ckpt`
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/htorre/Documents/anaconda/anaconda3/envs/P310/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:255: UserWarning: Found keys that are not in the model state dict b

**** Fin factualidad


In [39]:
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)

{'mean_precision': 0.8279218648543144, 'mean_recall': 0.8130444171875322, 'mean_f1': 0.8203009269482857, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cpu')}
{'n_examples': 379, 'lang': 'en', 'mean_automated_readability': 12.962654002661017, 'mean_coleman_liau': 13.544975841342726, 'mean_dale_chall': 11.042229873784924, 'mean_flesch_kincaid_grade': 11.490371639590998, 'mean_flesch_reading_ease': 43.18348842750842, 'mean_gunning_fog': 13.882925550125412, 'mean_num_sentences': 20.094986807387862, 'mean_num_words': 334.712401055409, 'mean_reading_time_sec': 26.894172928759897, 'mean_smog_index': 13.303621532167545, 'mean_syllables': 576.4353562005277, 'mean_text_standard': 12.546174142480211}
{'mean_alignscore': 0.3286242190954868, 'std_alignscore': 0.08548606122105906, 'min_alignscore': 0.11750971525907516, 'max_alignscore': 0.5550947189331055, 'n_examples': 379, 'backbone': 'roberta-base', 'evaluation_mode'

In [40]:
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_llama3.2-1b.csv", index=False)

### Calculo de ejemplo de las 3 metricas requeridas para llama3 Con CoT

In [7]:
data = pd.read_csv('models/results/summaries_llama32-1b_COT.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_llama32-1b_COT.csv", index=False)

**** Inicio relevancia


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


c:\ProgramData\anaconda3\envs\alignscore-113\lib\site-packages\lightning_fabric\__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file c:\Users\jsoa\Documents\GitHub\Proyecto-PLN-FLAG\src\models\alignscore\AlignScore-base.ckpt`
Some weights of RobertaM

**** Fin factualidad
{'mean_precision': 0.8583057787857558, 'mean_recall': 0.8500750455417131, 'mean_f1': 0.8539697460438076, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cuda')}
{'n_examples': 380, 'lang': 'en', 'mean_automated_readability': 12.448938174421633, 'mean_coleman_liau': 13.1186682583311, 'mean_dale_chall': 11.725205417514927, 'mean_flesch_kincaid_grade': 11.43344517620614, 'mean_flesch_reading_ease': 42.008626840077916, 'mean_gunning_fog': 14.431631104840006, 'mean_num_sentences': 29.676315789473684, 'mean_num_words': 462.38947368421054, 'mean_reading_time_sec': 37.48679247368421, 'mean_smog_index': 13.554977375983038, 'mean_syllables': 816.6157894736842, 'mean_text_standard': 12.51842105263158}
{'mean_alignscore': 0.6461376143129248, 'std_alignscore': 0.15044893288599906, 'min_alignscore': 0.2060336470603943, 'max_alignscore': 0.9646719694137573, 'n_examples': 380, 'backbone': 'roberta-base'

In [8]:
data = pd.read_csv('models/results/summaries_llama32-3b_COT.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_llama32-3b_COT.csv", index=False)

**** Inicio relevancia


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file c:\Users\jsoa\Documents\GitHub\Proyecto-PLN-FLAG\src\models\alignscore\AlignScore-base.ckpt`
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\ProgramData\anaconda3\envs\alignscore-113\lib\site-packages\pytorch_lightning\core\saving.py:255: UserWarning: Found keys that

**** Fin factualidad
{'mean_precision': 0.8559510337679009, 'mean_recall': 0.8510684861948615, 'mean_f1': 0.8533671526532424, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cuda')}
{'n_examples': 380, 'lang': 'en', 'mean_automated_readability': 11.757425276962907, 'mean_coleman_liau': 12.609185442872953, 'mean_dale_chall': 11.434504449546033, 'mean_flesch_kincaid_grade': 10.841409947213439, 'mean_flesch_reading_ease': 45.18866426837061, 'mean_gunning_fog': 13.657847340551065, 'mean_num_sentences': 34.113157894736844, 'mean_num_words': 517.8263157894737, 'mean_reading_time_sec': 41.355790552631575, 'mean_smog_index': 13.050943883704651, 'mean_syllables': 899.0052631578948, 'mean_text_standard': 12.021052631578947}
{'mean_alignscore': 0.6013612826403819, 'std_alignscore': 0.128130815454854, 'min_alignscore': 0.25775375962257385, 'max_alignscore': 0.941592812538147, 'n_examples': 380, 'backbone': 'roberta-base

### Calculo de ejemplo de las 3 metricas requeridas para gemma

In [7]:
data = pd.read_csv('models/results/summaries_gemma-3-1b-pt.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_gemma-3-1b-pt.csv", index=False)

**** Inicio relevancia


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


/Users/htorre/Documents/anaconda/anaconda3/envs/P310/lib/python3.10/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file models/alignscore/AlignScore-base.ckpt`
Some weights of RobertaModel were not initialized from the

**** Fin factualidad
{'mean_precision': 0.82111682938902, 'mean_recall': 0.8138819740006799, 'mean_f1': 0.8173210584803632, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cpu')}
{'n_examples': 380, 'lang': 'en', 'mean_automated_readability': 13.615634061159804, 'mean_coleman_liau': 14.088285805376556, 'mean_dale_chall': 11.466279532340204, 'mean_flesch_kincaid_grade': 11.819987138127718, 'mean_flesch_reading_ease': 40.55993701470229, 'mean_gunning_fog': 14.316384082759539, 'mean_num_sentences': 19.839473684210525, 'mean_num_words': 320.62105263157895, 'mean_reading_time_sec': 26.70564684210526, 'mean_smog_index': 13.536725684511072, 'mean_syllables': 565.2342105263158, 'mean_text_standard': 12.952631578947368}
{'mean_alignscore': 0.2831807450636437, 'std_alignscore': 0.0843669348098076, 'min_alignscore': 0.11659961938858032, 'max_alignscore': 0.6092774271965027, 'n_examples': 380, 'backbone': 'roberta-base'

### Calculo de ejemplo de las 3 metricas requeridas para gemma CoT

In [6]:
data = pd.read_csv('models/results/summaries_gemma3_COT.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_gemma3_COT.csv", index=False)

**** Inicio relevancia


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


c:\ProgramData\anaconda3\envs\alignscore-113\lib\site-packages\lightning_fabric\__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file c:\Users\jsoa\Documents\GitHub\Proyecto-PLN-FLAG\src\models\alignscore\AlignScore-base.ckpt`
Some weights of RobertaM

**** Fin factualidad
{'mean_precision': 0.8592638530229267, 'mean_recall': 0.8474098647895613, 'mean_f1': 0.852927167164652, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cuda')}
{'n_examples': 380, 'lang': 'en', 'mean_automated_readability': 13.37605063405537, 'mean_coleman_liau': 13.821419200545739, 'mean_dale_chall': 12.302120074314251, 'mean_flesch_kincaid_grade': 12.15568364672435, 'mean_flesch_reading_ease': 37.50566271374475, 'mean_gunning_fog': 15.315427032959832, 'mean_num_sentences': 25.692105263157895, 'mean_num_words': 396.2157894736842, 'mean_reading_time_sec': 32.951139, 'mean_smog_index': 14.118077710719707, 'mean_syllables': 717.6605263157895, 'mean_text_standard': 13.239473684210527}
{'mean_alignscore': 0.6733246902690122, 'std_alignscore': 0.1774761193741378, 'min_alignscore': 0.11249250918626785, 'max_alignscore': 0.9770880937576294, 'n_examples': 380, 'backbone': 'roberta-base', 'evalua

### Calculo de ejemplo de las 3 metricas requeridas para qwen

In [7]:
data = pd.read_csv('models/results/summaries_qwen3-lora.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_qwen3-lora.csv", index=False)

**** Inicio relevancia


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


c:\ProgramData\anaconda3\envs\alignscore-113\lib\site-packages\lightning_fabric\__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.8.0.post1 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file c:\Users\jsoa\Documents\GitHub\Proyecto-PLN-FLAG\src\models\alignscore\AlignScore-large.ckpt`
Some weights of R

**** Fin factualidad
{'mean_precision': 0.8464673945778295, 'mean_recall': 0.821945740517817, 'mean_f1': 0.8339219959158647, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cuda')}
{'n_examples': 380, 'lang': 'en', 'mean_automated_readability': 12.596478824074955, 'mean_coleman_liau': 13.234853655992794, 'mean_dale_chall': 11.531765226962087, 'mean_flesch_kincaid_grade': 11.69032582496056, 'mean_flesch_reading_ease': 41.07225075980072, 'mean_gunning_fog': 14.689156145683999, 'mean_num_sentences': 16.107894736842105, 'mean_num_words': 259.7605263157895, 'mean_reading_time_sec': 20.77707210526316, 'mean_smog_index': 13.706138915423255, 'mean_syllables': 455.2342105263158, 'mean_text_standard': 12.48157894736842}
{'mean_alignscore': 0.5731018154244674, 'std_alignscore': 0.11350389979424946, 'min_alignscore': 0.22866535186767578, 'max_alignscore': 0.8984176516532898, 'n_examples': 380, 'backbone': 'roberta-base'

### Calculo de ejemplo de las 3 metricas requeridas para qwen con CoT Implicito

In [11]:
data = pd.read_csv('models/results/summaries_qwen3_COT_imp.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_qwen3_COT_imp.csv", index=False)

**** Inicio relevancia


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


c:\ProgramData\anaconda3\envs\alignscore-113\lib\site-packages\lightning_fabric\__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file c:\Users\jsoa\Documents\GitHub\Proyecto-PLN-FLAG\src\models\alignscore\AlignScore-base.ckpt`
Some weights of RobertaM

**** Fin factualidad
{'mean_precision': 0.84110905653552, 'mean_recall': 0.8202082284187016, 'mean_f1': 0.8304590044837249, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cuda')}
{'n_examples': 380, 'lang': 'en', 'mean_automated_readability': 10.758226416687998, 'mean_coleman_liau': 12.126139290875667, 'mean_dale_chall': 10.863874985561802, 'mean_flesch_kincaid_grade': 9.950942268872625, 'mean_flesch_reading_ease': 49.175338361364865, 'mean_gunning_fog': 12.580707783214747, 'mean_num_sentences': 20.20263157894737, 'mean_num_words': 279.91052631578947, 'mean_reading_time_sec': 21.83065436842105, 'mean_smog_index': 12.229836526982961, 'mean_syllables': 472.7236842105263, 'mean_text_standard': 11.144736842105264}
{'mean_alignscore': 0.47348443434426657, 'std_alignscore': 0.09649911191847776, 'min_alignscore': 0.23216304183006287, 'max_alignscore': 0.7105181217193604, 'n_examples': 380, 'backbone': 'roberta-bas

### Calculo de ejemplo de las 3 metricas requeridas para phi3.5

In [ ]:
data = pd.read_csv('models/results/summaries_phi35.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_phi35.csv", index=False)

**** Inicio relevancia


/Users/jsoa/miniforge3/envs/alignscore-112/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.bias', 'lm_head.layer_norm.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.bias', 'lm_head.layer_norm.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file models/alignscore/AlignScore-base.ckpt`
Some weights of the model checkpoint at roberta-base were not used when initializing Ro

**** Fin factualidad
{'mean_precision': 0.8530013038923866, 'mean_recall': 0.837771927212414, 'mean_f1': 0.8450050744571184, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': 'cpu'}
{'n_examples': 380, 'lang': 'en', 'mean_automated_readability': 14.253421052631577, 'mean_coleman_liau': 15.244000000000003, 'mean_dale_chall': 9.989157894736842, 'mean_flesch_kincaid_grade': 10.433157894736842, 'mean_flesch_reading_ease': 49.641447368421055, 'mean_gunning_fog': 11.08286842105263, 'mean_num_sentences': 22.82894736842105, 'mean_num_words': 364.8, 'mean_reading_time_sec': 30.8485, 'mean_smog_index': 12.332786885245902, 'mean_syllables': 599.1578947368421, 'mean_text_standard': 11.260526315789473}
{'mean_alignscore': 0.45983980549009223, 'std_alignscore': 0.17386677445404883, 'min_alignscore': 0.1141374409198761, 'max_alignscore': 0.9775390028953552, 'n_examples': 380, 'backbone': 'roberta-base', 'evaluation_mode': 'nli_sp', 'batc

### Calculo de ejemplo de las 3 metricas requeridas para phi3.5 CoT

In [7]:
data = pd.read_csv('models/results/summaries_phi35_COT.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_phi35_COT.csv", index=False)

**** Inicio relevancia


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


c:\ProgramData\anaconda3\envs\alignscore-113\lib\site-packages\lightning_fabric\__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file c:\Users\jsoa\Documents\GitHub\Proyecto-PLN-FLAG\src\models\alignscore\AlignScore-base.ckpt`
Some weights of RobertaM

**** Fin factualidad
{'mean_precision': 0.8777736883414419, 'mean_recall': 0.8455527343248066, 'mean_f1': 0.8607283585949947, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cuda')}
{'n_examples': 380, 'lang': 'en', 'mean_automated_readability': 14.685149397396952, 'mean_coleman_liau': 15.166000113643259, 'mean_dale_chall': 12.635054934203827, 'mean_flesch_kincaid_grade': 13.328009948649543, 'mean_flesch_reading_ease': 30.40563701890857, 'mean_gunning_fog': 16.389684721999426, 'mean_num_sentences': 19.181578947368422, 'mean_num_words': 307.0921052631579, 'mean_reading_time_sec': 26.16934586842105, 'mean_smog_index': 14.682345765717796, 'mean_syllables': 572.85, 'mean_text_standard': 14.25}
{'mean_alignscore': 0.7550318036032351, 'std_alignscore': 0.16397291456206228, 'min_alignscore': 0.0911603569984436, 'max_alignscore': 0.9716159701347351, 'n_examples': 380, 'backbone': 'roberta-base', 'evaluation_mode': '